In [2]:
!pip install kaggle


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import sys

def setup_kaggle():
    if 'google.colab' in sys.modules:
        # Colab setup
        from google.colab import files
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        !mv kaggle.json /root/.kaggle/
        !chmod 600 /root/.kaggle/kaggle.json
    else:
        # Local Jupyter setup
        kaggle_path = os.path.expanduser("~/.kaggle/kaggle.json")
        if not os.path.exists(kaggle_path):
            raise FileNotFoundError("❌ kaggle.json not found in ~/.kaggle/")
        else:
            print("✅ kaggle.json found locally")

setup_kaggle()

✅ kaggle.json found locally


In [6]:
!dir C:\Users\User\.kaggle

 Volume in drive C has no label.
 Volume Serial Number is 6EDE-80BE

 Directory of C:\Users\User\.kaggle

10/20/2026  06:50 PM    <DIR>          .
10/20/2026  10:32 AM    <DIR>          ..
10/20/2026  10:43 AM                81 kaggle.json
               1 File(s)             81 bytes
               2 Dir(s)  165,059,792,896 bytes free


In [8]:
!kaggle --version

Kaggle CLI 2.2.3


## Dataset

In [7]:
!kaggle datasets list -s pneumonia

ref                                                            title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
paultimothymooney/chest-xray-pneumonia                         Chest X-Ray Images (Pneumonia)                       2463365435  2018-03-24 19:41:59.603000         681977       7750  0.75             
pcbreviglieri/pneumonia-xray-images                            Pneumonia X-Ray Images                               1225740412  2020-05-18 14:32:09.660000          13037        102  0.9375           
iamtapendu/rsna-pneumonia-processed-dataset                    RSNA Pneumonia Processed Dataset                    11748997093  2024-11-10 17:28:20.063000           5714         38  1                


## raw dir 

In [2]:
!mkdir data\raw

In [9]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p data/raw/

"""If authentication is required then run: !kaggle auth login"""

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token


In [12]:
!kaggle datasets list -s pneumonia

ref                                                            title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
paultimothymooney/chest-xray-pneumonia                         Chest X-Ray Images (Pneumonia)                       2463365435  2018-03-24 19:41:59.603000         681977       7750  0.75             
pcbreviglieri/pneumonia-xray-images                            Pneumonia X-Ray Images                               1225740412  2020-05-18 14:32:09.660000          13037        102  0.9375           
iamtapendu/rsna-pneumonia-processed-dataset                    RSNA Pneumonia Processed Dataset                    11748997093  2024-11-10 17:28:20.063000           5714         38  1                


## Unzip Dataset

In [14]:
import zipfile
import os

zip_path = r"C:\Users\User\Computer Vision Project\data\raw\chest-xray-pneumonia.zip"
extract_dir = r"C:\Users\User\Computer Vision Project\data\raw"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted to:", extract_dir)

Dataset extracted to: C:\Users\User\Computer Vision Project\data\raw


In [15]:
base_dir = r"C:\Users\User\Computer Vision Project\data\raw\chest_xray"
for folder in ["train","test","val"]:
    print(folder,":",os.listdir(os.path.join(base_dir, folder))[:5])

train : ['NORMAL', 'PNEUMONIA']
test : ['NORMAL', 'PNEUMONIA']
val : ['NORMAL', 'PNEUMONIA']


In [16]:
for folder in ["train","test","val"]:
    for category in ["NORMAL","PNEUMONIA"]:
        path = os.path.join(base_dir, folder, category)
        print(folder, category, ":", len(os.listdir(path)))
        

train NORMAL : 1341
train PNEUMONIA : 3875
test NORMAL : 234
test PNEUMONIA : 390
val NORMAL : 8
val PNEUMONIA : 8


## Merge valid folder to train

In [19]:

import shutil

base_dir = r"C:\Users\User\Computer Vision Project\data\raw\chest_xray"

for category in ["NORMAL", "PNEUMONIA"]:
    val_path = os.path.join(base_dir, "val", category)
    train_path = os.path.join(base_dir, "train", category)

    for file in os.listdir(val_path):
        src = os.path.join(val_path, file)
        dst = os.path.join(train_path, file)
        shutil.move(src, dst)

print("Merged val into train")

Merged val into train


## Interim dir

In [22]:
import os

base_dir = r"C:\Users\User\Computer Vision Project\data"
interim_dir = os.path.join(base_dir,"interim")

os.makedirs(interim_dir, exist_ok=True)
print("Interim folder created at:",interim_dir)


Interim folder created at: C:\Users\User\Computer Vision Project\data\interim


In [23]:
from torchvision import transforms

def save_dataset(dataset, split_name, dst_root):
    dst_folder = os.path.join(dst_root, split_name)
    os.makedirs(dst_folder, exist_ok=True)

    for idx, (tensor_img, label) in enumerate(dataset):
        # Convert tensor back to PIL
        pil_img = transforms.ToPILImage()(tensor_img)
        class_name = dataset.classes[label]
        class_dir = os.path.join(dst_folder, class_name)
        os.makedirs(class_dir, exist_ok=True)

        pil_img.save(os.path.join(class_dir, f"{split_name}_{idx}.png"))

    print(f"✅ Saved {len(dataset)} {split_name} images into interim")

# Save both splits
save_dataset(train_data, "train", interim_dir)
save_dataset(test_data, "test", interim_dir)


✅ Saved 5232 train images into interim
✅ Saved 624 test images into interim
